# Score calculations for all six embedding models

1. LogME score
1. H-Score
1. LFC

## LogME score

In [1]:
import os
import json
import numpy as np

In [10]:
def compute_logme(features: np.ndarray, labels: np.ndarray, max_iter: int = 25, tol: float = 1e-4):
    """
    Computes overall and per-class LogME scores with stable MacKay fixed-point updates.
    """
    N, D = features.shape
    unique_classes = np.unique(labels)
    num_classes = len(unique_classes)
    
    # 1. Mean-centre features
    F = features - np.mean(features, axis=0, keepdims=True)
    
    # 2. Thin SVD: F = U * diag(s) * V^T
    U, s, _ = np.linalg.svd(F, full_matrices=False)
    k = len(s)
    s2 = s ** 2

    class_scores = {}
    total_log_evidence = 0.0

    for c in unique_classes:
        # Binary target centred to remove DC bias
        y = (labels == c).astype(np.float64)
        y_centered = y - np.mean(y)
        
        y_proj = U.T @ y_centered          # Shape: (k,)
        y_norm_sq = np.dot(y_centered, y_centered)
        y_proj_sq = np.sum(y_proj ** 2)
        y_ortho_sq = max(0.0, y_norm_sq - y_proj_sq)  # Energy outside feature span

        # Initialise hyperparameters
        alpha = 1.0
        beta = 1.0 / (np.var(y_centered) + 1e-4)

        for _ in range(max_iter):
            prev_alpha, prev_beta = alpha, beta
            
            denom = alpha + beta * s2
            gamma = np.sum((beta * s2) / denom)
            
            m = (beta * s / denom) * y_proj
            w_norm_sq = np.sum(m ** 2)
            
            rss = y_ortho_sq + np.sum(((alpha / denom) * y_proj) ** 2)

            alpha = gamma / (w_norm_sq + 1e-12)
            beta = max(1e-12, N - gamma) / (rss + 1e-12)

            if abs(alpha - prev_alpha) / (alpha + 1e-12) < tol and abs(beta - prev_beta) / (beta + 1e-12) < tol:
                break

        # Log determinant: log |C|
        log_det = -N * np.log(beta + 1e-12) - k * np.log(alpha + 1e-12) + np.sum(np.log(alpha + beta * s2 + 1e-12))
        
        # Quadratic form: y^T C^{-1} y
        quad_form = beta * y_ortho_sq + np.sum((alpha * beta / (alpha + beta * s2 + 1e-12)) * (y_proj ** 2))
        
        # Marginal log likelihood for class c
        log_evidence_c = -0.5 * (N * np.log(2.0 * np.pi) + log_det + quad_form)
        
        class_score = float(log_evidence_c / N)
        class_scores[int(c)] = class_score
        total_log_evidence += log_evidence_c

    overall_score = float(total_log_evidence / (N * num_classes))
    return overall_score, class_scores


def get_first_k_subset(features: np.ndarray, labels: np.ndarray, shots_per_class: int):
    """
    Deterministically selects the first `shots_per_class` samples for each class
    in the order they appear in the archive.
    """
    selected_indices = []

    for c in np.unique(labels):
        c_indices = np.where(labels == c)[0]
        if len(c_indices) < shots_per_class:
            raise ValueError(f"Class {c} only has {len(c_indices)} samples; cannot pick {shots_per_class}.")
        
        # Take the first k sample indices deterministically
        selected_indices.extend(c_indices[:shots_per_class])

    selected_indices = np.array(selected_indices)
    return features[selected_indices], labels[selected_indices]


# --- Configuration ---
SHOTS_PER_CLASS =  25  # Set to 10, 5, or 1

base_dir = "./model_embeddings_train"
output_dir = f"./model_scores/model_scores_{SHOTS_PER_CLASS}_shot/logME"
os.makedirs(output_dir, exist_ok=True)

results = {}

for model_name in sorted(os.listdir(base_dir)):
    model_path = os.path.join(base_dir, model_name)
    if not os.path.isdir(model_path):
        continue
        
    npz_path = os.path.join(model_path, "features.npz")
    if not os.path.exists(npz_path):
        continue
        
    print(f"Evaluating {model_name} on first {SHOTS_PER_CLASS} samples/class...")
    data = np.load(npz_path)
    full_features = data["embeddings"]
    full_labels = data["labels"]
    
    # Deterministic subset extraction
    features, labels = get_first_k_subset(
        full_features, 
        full_labels, 
        shots_per_class=SHOTS_PER_CLASS
    )
    
    overall, class_wise = compute_logme(features, labels)
    
    results[model_name] = {
        "shots_per_class": SHOTS_PER_CLASS,
        "total_samples": int(features.shape[0]),
        "overall_logme": overall,
        "class_wise_logme": class_wise
    }

output_file = os.path.join(output_dir, "all_models_logme.json")
with open(output_file, "w") as f:
    json.dump(results, f, indent=4)

print(f"\nSaved deterministic {SHOTS_PER_CLASS}-shot LogME scores to {output_file}")

Evaluating AVES-bio on first 25 samples/class...
Evaluating BirdAVES-biox-base on first 25 samples/class...
Evaluating DistilHuBERT on first 25 samples/class...
Evaluating HuBERT on first 25 samples/class...
Evaluating MusicHuBERT on first 25 samples/class...
Evaluating WavLM on first 25 samples/class...

Saved deterministic 25-shot LogME scores to ./model_scores/model_scores_25_shot/logME\all_models_logme.json


## H score

In [16]:
import os
import json
import numpy as np

def compute_hscore(features: np.ndarray, labels: np.ndarray, eps: float = 1e-4):
    """
    Computes overall and per-class H-scores for features (N, D) and labels (N,).
    """
    N, D = features.shape
    unique_classes, counts = np.unique(labels, return_counts=True)
    
    # 1. Mean-centre features across samples
    F_centered = features - np.mean(features, axis=0, keepdims=True)

    # 2. Total feature covariance matrix with Tikhonov regularisation
    Sigma_f = (F_centered.T @ F_centered) / N
    Sigma_f_reg = Sigma_f + eps * np.eye(D)
    Sigma_f_inv = np.linalg.pinv(Sigma_f_reg)

    # 3. Class-wise centroid Mahalanobis evaluation
    class_scores = {}
    overall_hscore = 0.0

    for c, count in zip(unique_classes, counts):
        idx = np.where(labels == c)[0]
        mu_c = np.mean(F_centered[idx], axis=0)  # Class-conditional mean vector (D,)
        
        # Squared Mahalanobis distance of class centroid
        h_c = float(mu_c.T @ Sigma_f_inv @ mu_c)
        class_scores[int(c)] = h_c
        
        # Prior-weighted contribution to the overall trace
        overall_hscore += (count / N) * h_c

    return float(overall_hscore), class_scores


def get_first_k_subset(features: np.ndarray, labels: np.ndarray, shots_per_class: int):
    """
    Deterministically selects the first `shots_per_class` samples for each class
    in the order they appear in the archive.
    """
    selected_indices = []

    for c in np.unique(labels):
        c_indices = np.where(labels == c)[0]
        if len(c_indices) < shots_per_class:
            raise ValueError(f"Class {c} only has {len(c_indices)} samples; cannot pick {shots_per_class}.")
        
        # Take the first k sample indices deterministically
        selected_indices.extend(c_indices[:shots_per_class])

    selected_indices = np.array(selected_indices)
    return features[selected_indices], labels[selected_indices]


# --- Configuration ---
SHOTS_PER_CLASS = 25   # Set to 10, 5, or 1

base_dir = "./model_embeddings_train"
output_dir = f"./model_scores/model_scores_{SHOTS_PER_CLASS}_shot/H_score"
os.makedirs(output_dir, exist_ok=True)

results = {}

for model_name in sorted(os.listdir(base_dir)):
    model_path = os.path.join(base_dir, model_name)
    if not os.path.isdir(model_path):
        continue
        
    npz_path = os.path.join(model_path, "features.npz")
    if not os.path.exists(npz_path):
        continue
        
    print(f"Evaluating H-score for {model_name} on first {SHOTS_PER_CLASS} samples/class...")
    data = np.load(npz_path)
    full_features = data["embeddings"]
    full_labels = data["labels"]
    
    # Deterministic subset extraction
    features, labels = get_first_k_subset(
        full_features, 
        full_labels, 
        shots_per_class=SHOTS_PER_CLASS
    )
    
    overall, class_wise = compute_hscore(features, labels)
    
    results[model_name] = {
        "shots_per_class": SHOTS_PER_CLASS,
        "total_samples": int(features.shape[0]),
        "overall_hscore": overall,
        "class_wise_hscore": class_wise
    }

output_file = os.path.join(output_dir, "all_models_hscore.json")
with open(output_file, "w") as f:
    json.dump(results, f, indent=4)

print(f"\nSaved deterministic {SHOTS_PER_CLASS}-shot H-scores to {output_file}")

Evaluating H-score for AVES-bio on first 25 samples/class...
Evaluating H-score for BirdAVES-biox-base on first 25 samples/class...
Evaluating H-score for DistilHuBERT on first 25 samples/class...
Evaluating H-score for HuBERT on first 25 samples/class...
Evaluating H-score for MusicHuBERT on first 25 samples/class...
Evaluating H-score for WavLM on first 25 samples/class...

Saved deterministic 25-shot H-scores to ./model_scores/model_scores_25_shot/H_score\all_models_hscore.json


# NTK - LFC

In [22]:
import os
import json
import numpy as np

def compute_ntk_lfc(features: np.ndarray, labels: np.ndarray, centered: bool = True):
    """
    Computes overall and class-wise NTK LFC with deterministic, consistent decomposition.
    """
    N, D = features.shape
    unique_classes, counts = np.unique(labels, return_counts=True)
    num_classes = len(unique_classes)
    
    # 1. Unit-norm normalise embeddings
    norms = np.linalg.norm(features, axis=1, keepdims=True) + 1e-8
    F_norm = features / norms

    # 2. Linear NTK / Gram Matrix
    K = np.clip(F_norm @ F_norm.T, -1.0, 1.0)
    
    # Construct ideal block-diagonal label kernel K_Y
    Y_onehot = np.eye(num_classes)[labels]  # (N, C)
    K_Y = Y_onehot @ Y_onehot.T             # (N, N)

    # 3. Kernel Centering: H * K * H, where H = I - (1/N) 1 1^T
    if centered:
        H = np.eye(N) - (1.0 / N) * np.ones((N, N))
        K = H @ K @ H
        K_Y = H @ K_Y @ H

    norm_K = np.linalg.norm(K, 'fro') + 1e-12
    norm_KY = np.linalg.norm(K_Y, 'fro') + 1e-12

    # 4. Class-wise decomposition
    class_scores = {}
    total_alignment = 0.0

    for c in unique_classes:
        y_c = (labels == c).astype(np.float64)
        
        if centered:
            # Centred target vector
            y_c_tilde = y_c - np.mean(y_c)
            intra_c = float(y_c_tilde.T @ K @ y_c_tilde)
        else:
            idx = np.where(labels == c)[0]
            intra_c = float(np.sum(K[np.ix_(idx, idx)]))

        score_c = intra_c / (norm_K * norm_KY)
        class_scores[int(c)] = score_c
        total_alignment += score_c

    overall_lfc = float(total_alignment)
    return overall_lfc, class_scores


def get_first_k_subset(features: np.ndarray, labels: np.ndarray, shots_per_class: int):
    """
    Deterministically selects the first `shots_per_class` samples for each class
    in the order they appear in the archive.
    """
    selected_indices = []

    for c in np.unique(labels):
        c_indices = np.where(labels == c)[0]
        if len(c_indices) < shots_per_class:
            raise ValueError(f"Class {c} only has {len(c_indices)} samples; cannot pick {shots_per_class}.")
        
        selected_indices.extend(c_indices[:shots_per_class])

    selected_indices = np.array(selected_indices)
    return features[selected_indices], labels[selected_indices]


# --- Configuration ---
SHOTS_PER_CLASS = 25   # Set to 10, 5, or 1

base_dir = "./model_embeddings_train"
output_dir = f"./model_scores/model_scores_{SHOTS_PER_CLASS}_shot/LFC_score"
os.makedirs(output_dir, exist_ok=True)

results = {}

for model_name in sorted(os.listdir(base_dir)):
    model_path = os.path.join(base_dir, model_name)
    if not os.path.isdir(model_path):
        continue
        
    npz_path = os.path.join(model_path, "features.npz")
    if not os.path.exists(npz_path):
        continue
        
    print(f"Evaluating LFC for {model_name} on first {SHOTS_PER_CLASS} samples/class...")
    data = np.load(npz_path)
    full_features = data["embeddings"]
    full_labels = data["labels"]
    
    # Deterministic subset extraction
    features, labels = get_first_k_subset(
        full_features, 
        full_labels, 
        shots_per_class=SHOTS_PER_CLASS
    )
    
    overall, class_wise = compute_ntk_lfc(features, labels, centered=True)
    
    results[model_name] = {
        "shots_per_class": SHOTS_PER_CLASS,
        "total_samples": int(features.shape[0]),
        "overall_lfc": overall,
        "class_wise_lfc": class_wise
    }

output_file = os.path.join(output_dir, "all_models_lfc.json")
with open(output_file, "w") as f:
    json.dump(results, f, indent=4)

print(f"\nSaved deterministic {SHOTS_PER_CLASS}-shot centered LFC scores to {output_file}")

Evaluating LFC for AVES-bio on first 25 samples/class...
Evaluating LFC for BirdAVES-biox-base on first 25 samples/class...
Evaluating LFC for DistilHuBERT on first 25 samples/class...
Evaluating LFC for HuBERT on first 25 samples/class...
Evaluating LFC for MusicHuBERT on first 25 samples/class...
Evaluating LFC for WavLM on first 25 samples/class...

Saved deterministic 25-shot centered LFC scores to ./model_scores/model_scores_25_shot/LFC_score\all_models_lfc.json


## RKHS score

In [1]:
import os
import json
import numpy as np

In [6]:
def compute_linear_rkhs(
    features: np.ndarray,
    labels: np.ndarray,
    num_classes: int = 6,
    lam: float = 1.0,
):
    """
    Computes double-centered, scale-invariant negative RKHS Label Complexity Norm.
    """
    N, _ = features.shape

    # 1. L2-normalize features
    norms = np.linalg.norm(features, axis=1, keepdims=True) + 1e-8
    features_norm = features / norms

    # 2. Compute and double-center empirical NTK (removes shared anisotropy cone)
    Theta = (features_norm @ features_norm.T).astype(np.float64)
    H = np.eye(N, dtype=np.float64) - (1.0 / N) * np.ones((N, N), dtype=np.float64)
    Theta_tilde = H @ Theta @ H

    # 3. Trace-normalize the centered kernel so tr(Theta_bar) = N
    trace_val = np.trace(Theta_tilde)
    Theta_bar = (N / (trace_val + 1e-12)) * Theta_tilde

    # 4. Centered one-hot label matrix (N x C)
    unique_classes, mapped_labels = np.unique(labels, return_inverse=True)
    Y_onehot = np.eye(num_classes, dtype=np.float64)[mapped_labels]
    Y_tilde = Y_onehot - (1.0 / num_classes)

    # 5. Solve (Theta_bar + lam * I) A = Y_tilde
    Theta_reg = Theta_bar + lam * np.eye(N, dtype=np.float64)
    A = np.linalg.solve(Theta_reg, Y_tilde)

    # 6. Per-class and overall negative RKHS complexity
    class_scores = {}
    overall_rkhs = 0.0

    for idx, c in enumerate(unique_classes):
        rkhs_c = -float(np.dot(Y_tilde[:, idx], A[:, idx])) / N
        class_scores[int(c)] = rkhs_c
        overall_rkhs += rkhs_c

    return float(overall_rkhs), class_scores


def get_first_k_subset(features: np.ndarray, labels: np.ndarray, shots_per_class: int):
    """
    Deterministically selects the first `shots_per_class` samples for each class
    in the order they appear in the archive.
    """
    selected_indices = []

    for c in np.unique(labels):
        c_indices = np.where(labels == c)[0]
        if len(c_indices) < shots_per_class:
            raise ValueError(
                f"Class {c} only has {len(c_indices)} samples; cannot pick {shots_per_class}."
            )
        selected_indices.extend(c_indices[:shots_per_class])

    selected_indices = np.array(selected_indices)
    return features[selected_indices], labels[selected_indices]


# --- Configuration ---
SHOTS_LIST = [1, 5, 10, 15, 20, 25]
NUM_CLASSES = 6
RIDGE_LAMBDA_MAP = {
    1: 1,
    5: 1,
    10: 0.5,
    15: 0.3,
    20: 0.2,
    25: 0.1
}

base_dir = "./model_embeddings_train" if os.path.exists("./model_embeddings_train") else "../model_embeddings_train"
scores_root = "./model_scores" if os.path.exists("./model_embeddings_train") else "../model_scores"

for shots in SHOTS_LIST:
    output_dir = os.path.join(scores_root, f"model_scores_{shots}_shot", "RKHS_linear")
    os.makedirs(output_dir, exist_ok=True)

    results = {}
    print(f"\nEvaluating Linear RKHS score ({shots}-shot)...")

    for model_name in sorted(os.listdir(base_dir)):
        model_path = os.path.join(base_dir, model_name)
        if not os.path.isdir(model_path):
            continue

        npz_path = os.path.join(model_path, "features.npz")
        if not os.path.exists(npz_path):
            continue

        data = np.load(npz_path)
        full_features = data["embeddings"]
        full_labels = data["labels"]

        try:
            features, labels = get_first_k_subset(
                full_features,
                full_labels,
                shots_per_class=shots,
            )
        except ValueError as e:
            print(f"  Skipping {model_name}: {e}")
            continue

        overall, class_wise = compute_linear_rkhs(
            features,
            labels,
            num_classes=NUM_CLASSES,
            lam=RIDGE_LAMBDA_MAP.get(shots, 1),
        )

        results[model_name] = {
            "shots_per_class": shots,
            "total_samples": int(features.shape[0]),
            "overall_rkhs": overall,
            "class_wise_rkhs": class_wise,
        }

    output_file = os.path.join(output_dir, "all_models_rkhs.json")
    with open(output_file, "w") as f:
        json.dump(results, f, indent=4)

    print(f"Saved deterministic {shots}-shot Linear RKHS scores to {output_file}")


Evaluating Linear RKHS score (1-shot)...
Saved deterministic 1-shot Linear RKHS scores to ./model_scores\model_scores_1_shot\RKHS_linear\all_models_rkhs.json

Evaluating Linear RKHS score (5-shot)...
Saved deterministic 5-shot Linear RKHS scores to ./model_scores\model_scores_5_shot\RKHS_linear\all_models_rkhs.json

Evaluating Linear RKHS score (10-shot)...
Saved deterministic 10-shot Linear RKHS scores to ./model_scores\model_scores_10_shot\RKHS_linear\all_models_rkhs.json

Evaluating Linear RKHS score (15-shot)...
Saved deterministic 15-shot Linear RKHS scores to ./model_scores\model_scores_15_shot\RKHS_linear\all_models_rkhs.json

Evaluating Linear RKHS score (20-shot)...
Saved deterministic 20-shot Linear RKHS scores to ./model_scores\model_scores_20_shot\RKHS_linear\all_models_rkhs.json

Evaluating Linear RKHS score (25-shot)...
Saved deterministic 25-shot Linear RKHS scores to ./model_scores\model_scores_25_shot\RKHS_linear\all_models_rkhs.json


## LGC CLS (Cross Entropy Loss)

In [7]:
import os
import json
import numpy as np

In [8]:
def _softmax(logits: np.ndarray) -> np.ndarray:
    shifted = logits - np.max(logits, axis=1, keepdims=True)
    exp_logits = np.exp(shifted)
    return exp_logits / np.sum(exp_logits, axis=1, keepdims=True)


def compute_linear_classification_ntk(
    features: np.ndarray,
    labels: np.ndarray,
    num_classes: int = 6,
    num_seeds: int = 10,
    normalize_features: bool = False,
):
    """
    Computes exact closed-form Classification-NTK scores (Plenk et al.) for a Linear Probe:
      1. overall_lgc_cls & class_wise_lgc_cls: Centered KTA of the probability-space NTK
         K_cls(i, j) = (z_i^T z_j + 1) * tr(H(p_i) H(p_j)) against centered one-hot labels.
      2. ce_grad_coherence: Normalized coherence of Cross-Entropy parameter gradients
         K_CE(i, j) = (z_i^T z_j + 1) * (p_i - y_i)^T (p_j - y_j).
    """
    N, D = features.shape
    X = features.astype(np.float64)

    if normalize_features:
        norms = np.linalg.norm(X, axis=1, keepdims=True) + 1e-8
        X = X / norms

    unique_classes, mapped_labels = np.unique(labels, return_inverse=True)
    Y_onehot = np.eye(num_classes, dtype=np.float64)[mapped_labels]
    Y_tilde = Y_onehot - (1.0 / num_classes)

    # Base linear logit kernel k_0(z_i, z_j) = z_i^T z_j + 1
    k0 = (X @ X.T) + 1.0

    # Match PyTorch nn.Linear(D, num_classes) uniform init bound: [-1/sqrt(D), 1/sqrt(D)]
    bound = 1.0 / np.sqrt(D)

    K_cls_sum = np.zeros((N, N), dtype=np.float64)
    K_ce_sum = np.zeros((N, N), dtype=np.float64)

    for seed in range(num_seeds):
        rng = np.random.default_rng(seed)
        W = rng.uniform(-bound, bound, size=(D, num_classes))
        b = rng.uniform(-bound, bound, size=(1, num_classes))

        logits = X @ W + b
        P = _softmax(logits)  # (N, C)

        # Softmax Jacobians H(p_i) = diag(p_i) - p_i p_i^T, shape (N, C, C)
        H = np.einsum("ni,ij->nij", P, np.eye(num_classes, dtype=np.float64)) - np.einsum(
            "ni,nj->nij", P, P
        )

        # Exact closed-form trace: tr(H(p_i) H(p_j))
        trace_HH = np.einsum("iac,jac->ij", H, H)
        K_cls_sum += k0 * trace_HH

        # Exact CE loss-gradient kernel: (p_i - y_i)^T (p_j - y_j) * k_0(z_i, z_j)
        residuals = P - Y_onehot  # (N, C)
        K_ce_sum += k0 * (residuals @ residuals.T)

    K_cls = K_cls_sum / num_seeds
    K_ce = K_ce_sum / num_seeds

    # --- 1. Centered Kernel Target Alignment (LGC_cls) ---
    H_N = np.eye(N, dtype=np.float64) - (1.0 / N) * np.ones((N, N), dtype=np.float64)
    K_Y = Y_tilde @ Y_tilde.T

    K_cls_centered = H_N @ K_cls @ H_N
    K_Y_centered = H_N @ K_Y @ H_N

    norm_cls = np.linalg.norm(K_cls_centered, ord="fro")
    norm_Y = np.linalg.norm(K_Y_centered, ord="fro")
    denom = (norm_cls * norm_Y) + 1e-12

    class_scores = {}
    overall_lgc_cls = 0.0
    Y_centered_cols = H_N @ Y_tilde  # (N, C)

    for idx, c in enumerate(unique_classes):
        y_col = Y_centered_cols[:, idx]
        score_c = float(y_col.T @ K_cls_centered @ y_col) / denom
        class_scores[int(c)] = score_c
        overall_lgc_cls += score_c

    # --- 2. Cross-Entropy Gradient Coherence ---
    norm_ce = np.linalg.norm(K_ce, ord="fro") + 1e-12
    ce_grad_coherence = float(np.sum(K_ce)) / (np.sqrt(N) * norm_ce)

    return float(overall_lgc_cls), class_scores, float(ce_grad_coherence)


def get_first_k_subset(features: np.ndarray, labels: np.ndarray, shots_per_class: int):
    selected_indices = []
    for c in np.unique(labels):
        c_indices = np.where(labels == c)[0]
        if len(c_indices) < shots_per_class:
            raise ValueError(
                f"Class {c} only has {len(c_indices)} samples; cannot pick {shots_per_class}."
            )
        selected_indices.extend(c_indices[:shots_per_class])

    selected_indices = np.array(selected_indices)
    return features[selected_indices], labels[selected_indices]


if __name__ == "__main__":
    SHOTS_LIST = [1, 5, 10, 15, 20, 25]
    NUM_CLASSES = 6
    NUM_SEEDS = 10
    NORMALIZE_FEATURES = False

    base_dir = (
        "./model_embeddings_train"
        if os.path.exists("./model_embeddings_train")
        else "../model_embeddings_train"
    )
    scores_root = (
        "./model_scores"
        if os.path.exists("./model_embeddings_train")
        else "../model_scores"
    )

    for shots in SHOTS_LIST:
        output_dir = os.path.join(scores_root, f"model_scores_{shots}_shot", "LGC_cls_linear")
        os.makedirs(output_dir, exist_ok=True)

        results = {}
        print(f"\nEvaluating Linear Classification-NTK ({shots}-shot)...")

        for model_name in sorted(os.listdir(base_dir)):
            model_path = os.path.join(base_dir, model_name)
            if not os.path.isdir(model_path):
                continue

            npz_path = os.path.join(model_path, "features.npz")
            if not os.path.exists(npz_path):
                continue

            data = np.load(npz_path)
            features, labels = get_first_k_subset(
                data["embeddings"], data["labels"], shots_per_class=shots
            )

            overall_lgc_cls, class_wise, ce_coherence = compute_linear_classification_ntk(
                features=features,
                labels=labels,
                num_classes=NUM_CLASSES,
                num_seeds=NUM_SEEDS,
                normalize_features=NORMALIZE_FEATURES,
            )

            results[model_name] = {
                "shots_per_class": shots,
                "total_samples": int(features.shape[0]),
                "overall_lgc_cls": overall_lgc_cls,
                "ce_grad_coherence": ce_coherence,
                "class_wise_lgc_cls": class_wise,
            }

        output_file = os.path.join(output_dir, "all_models_lgc_cls.json")
        with open(output_file, "w") as f:
            json.dump(results, f, indent=4)

        print(f"Saved {shots}-shot scores to {output_file}")


Evaluating Linear Classification-NTK (1-shot)...
Saved 1-shot scores to ./model_scores\model_scores_1_shot\LGC_cls_linear\all_models_lgc_cls.json

Evaluating Linear Classification-NTK (5-shot)...
Saved 5-shot scores to ./model_scores\model_scores_5_shot\LGC_cls_linear\all_models_lgc_cls.json

Evaluating Linear Classification-NTK (10-shot)...
Saved 10-shot scores to ./model_scores\model_scores_10_shot\LGC_cls_linear\all_models_lgc_cls.json

Evaluating Linear Classification-NTK (15-shot)...
Saved 15-shot scores to ./model_scores\model_scores_15_shot\LGC_cls_linear\all_models_lgc_cls.json

Evaluating Linear Classification-NTK (20-shot)...
Saved 20-shot scores to ./model_scores\model_scores_20_shot\LGC_cls_linear\all_models_lgc_cls.json

Evaluating Linear Classification-NTK (25-shot)...
Saved 25-shot scores to ./model_scores\model_scores_25_shot\LGC_cls_linear\all_models_lgc_cls.json


## SVM margin and SVM LOO margin

In [9]:
import os
import json
import itertools
import numpy as np
from sklearn.svm import SVC

In [11]:


def _fit_binary_precomputed_svm(K_sub: np.ndarray, y_sub: np.ndarray, C_svm: float = 1.0):
    """
    Fits a binary SVM on a precomputed PSD kernel submatrix K_sub (M x M)
    with labels y_sub in {-1, +1}.

    Returns:
        dual_obj: Exact Wolfe dual objective W(alpha*) = sum(alpha) - 0.5 * ||f||_H^2
        sum_alpha: Sum of dual variables sum(alpha_i) (= 1 / gamma^2 when separable)
        norm_f: RKHS norm ||f*||_H
        sv_indices: Indices of support vectors relative to K_sub
        y_alpha: Signed dual coefficients (y_i * alpha_i) at support vectors
        intercept: Optimal bias b*
    """
    clf = SVC(kernel="precomputed", C=C_svm)
    clf.fit(K_sub, y_sub)

    sv_indices = clf.support_
    y_alpha = clf.dual_coef_[0]  # y_i * alpha_i for i in sv_indices
    sum_alpha = float(np.sum(np.abs(y_alpha)))

    K_sv = K_sub[np.ix_(sv_indices, sv_indices)]
    norm_sq = float(y_alpha @ K_sv @ y_alpha)
    norm_f = float(np.sqrt(max(norm_sq, 1e-12)))

    dual_obj = sum_alpha - 0.5 * norm_sq
    intercept = float(clf.intercept_[0])

    return dual_obj, sum_alpha, norm_f, sv_indices, y_alpha, intercept


def compute_linear_svm_margin_scores(
    features: np.ndarray,
    labels: np.ndarray,
    num_classes: int = 6,
    C_svm: float = 1.0,
):
    """
    Computes two Kernel SVM Margin transferability scores for a Linear Probe
    using the double-centered, trace-normalized linear NTK Theta_0_bar:

      1. overall_svm_margin: Negative mean One-vs-One Wolfe dual objective
         -1 / (C choose 2) * sum_{c1 < c2} W^{(c1, c2)}(alpha*),
         which equals -1 / (2 * gamma^2) in the separable regime.
      2. overall_loo_margin: Mean Leave-One-Out (LOO) multi-class normalized
         geometric margin across all N samples (defined for K >= 2; None at K = 1).
    """
    N, _ = features.shape

    # 1. L2-normalize features
    norms = np.linalg.norm(features, axis=1, keepdims=True) + 1e-8
    X_norm = features / norms

    # 2. Linear probe NTK: Theta_0 = X_norm @ X_norm.T + 1
    Theta_0 = (X_norm @ X_norm.T).astype(np.float64) + 1.0

    # 3. Double-center and trace-normalize so tr(Theta_bar) = N (scale invariance)
    H_N = np.eye(N, dtype=np.float64) - (1.0 / N) * np.ones((N, N), dtype=np.float64)
    Theta_tilde = H_N @ Theta_0 @ H_N
    trace_val = np.trace(Theta_tilde)
    Theta_bar = (N / (trace_val + 1e-12)) * Theta_tilde

    # Small diagonal jitter for numerical stability in QP solver
    Theta_bar = Theta_bar + 1e-9 * np.eye(N, dtype=np.float64)

    unique_classes = np.unique(labels)
    if len(unique_classes) != num_classes:
        raise ValueError(f"Expected {num_classes} classes, got {len(unique_classes)}.")

    class_indices = {c: np.where(labels == c)[0] for c in unique_classes}
    shots_per_class = len(class_indices[unique_classes[0]])

    # --- Score 1: In-Sample OvO Theta_0-SVM Margin ---
    pair_dual_objs = []
    pair_sum_alphas = []
    class_margin_accum = {int(c): [] for c in unique_classes}

    for c1, c2 in itertools.combinations(unique_classes, 2):
        idx1 = class_indices[c1]
        idx2 = class_indices[c2]
        pair_idx = np.concatenate([idx1, idx2])
        y_pair = np.concatenate([
            np.ones(len(idx1), dtype=np.float64),
            -np.ones(len(idx2), dtype=np.float64),
        ])

        K_pair = Theta_bar[np.ix_(pair_idx, pair_idx)]
        dual_obj, sum_alpha, _, _, _, _ = _fit_binary_precomputed_svm(
            K_pair, y_pair, C_svm=C_svm
        )

        pair_dual_objs.append(dual_obj)
        pair_sum_alphas.append(sum_alpha)
        class_margin_accum[int(c1)].append(-dual_obj)
        class_margin_accum[int(c2)].append(-dual_obj)

    overall_svm_margin = -float(np.mean(pair_dual_objs))
    overall_neg_sum_alpha = -float(np.mean(pair_sum_alphas))
    class_wise_svm_margin = {
        c: float(np.mean(vals)) for c, vals in class_margin_accum.items()
    }

    # --- Score 2: Leave-One-Out (LOO) Normalized Geometric Margin (K >= 2) ---
    if shots_per_class < 2:
        overall_loo_margin = None
        class_wise_loo_margin = None
    else:
        loo_margins = []
        class_loo_accum = {int(c): [] for c in unique_classes}

        for i in range(N):
            y_i = labels[i]
            idx_pos = np.array([idx for idx in class_indices[y_i] if idx != i])
            pairwise_margins_i = []

            for c_other in unique_classes:
                if c_other == y_i:
                    continue
                idx_neg = class_indices[c_other]
                train_idx = np.concatenate([idx_pos, idx_neg])
                y_train = np.concatenate([
                    np.ones(len(idx_pos), dtype=np.float64),
                    -np.ones(len(idx_neg), dtype=np.float64),
                ])

                K_train = Theta_bar[np.ix_(train_idx, train_idx)]
                _, _, norm_f, sv_rel, y_alpha, intercept = _fit_binary_precomputed_svm(
                    K_train, y_train, C_svm=C_svm
                )

                # Evaluate signed geometric margin of held-out sample i (+1 class)
                sv_global = train_idx[sv_rel]
                f_i = float(y_alpha @ Theta_bar[sv_global, i]) + intercept
                geom_margin_i = f_i / norm_f
                pairwise_margins_i.append(geom_margin_i)

            # Worst-case margin against closest competing class
            worst_case_margin_i = float(np.min(pairwise_margins_i))
            loo_margins.append(worst_case_margin_i)
            class_loo_accum[int(y_i)].append(worst_case_margin_i)

        overall_loo_margin = float(np.mean(loo_margins))
        class_wise_loo_margin = {
            c: float(np.mean(vals)) for c, vals in class_loo_accum.items()
        }

    return (
        overall_svm_margin,
        overall_neg_sum_alpha,
        class_wise_svm_margin,
        overall_loo_margin,
        class_wise_loo_margin,
    )


def get_first_k_subset(features: np.ndarray, labels: np.ndarray, shots_per_class: int):
    selected_indices = []
    for c in np.unique(labels):
        c_indices = np.where(labels == c)[0]
        if len(c_indices) < shots_per_class:
            raise ValueError(
                f"Class {c} only has {len(c_indices)} samples; cannot pick {shots_per_class}."
            )
        selected_indices.extend(c_indices[:shots_per_class])

    selected_indices = np.array(selected_indices)
    return features[selected_indices], labels[selected_indices]


if __name__ == "__main__":
    SHOTS_LIST = [1, 5, 10, 15, 20, 25]
    NUM_CLASSES = 6
    C_SVM = 1.0

    base_dir = (
        "./model_embeddings_train"
        if os.path.exists("./model_embeddings_train")
        else "../model_embeddings_train"
    )
    scores_root = (
        "./model_scores"
        if os.path.exists("./model_embeddings_train")
        else "../model_scores"
    )

    for shots in SHOTS_LIST:
        output_dir = os.path.join(
            scores_root, f"model_scores_{shots}_shot", "SVM_margin_linear"
        )
        os.makedirs(output_dir, exist_ok=True)

        results = {}
        print(f"\nEvaluating Linear Theta_0-SVM Margin ({shots}-shot)...")

        for model_name in sorted(os.listdir(base_dir)):
            model_path = os.path.join(base_dir, model_name)
            if not os.path.isdir(model_path):
                continue

            npz_path = os.path.join(model_path, "features.npz")
            if not os.path.exists(npz_path):
                continue

            data = np.load(npz_path)
            features, labels = get_first_k_subset(
                data["embeddings"], data["labels"], shots_per_class=shots
            )

            (
                overall_margin,
                overall_neg_sum_alpha,
                class_wise_margin,
                overall_loo_margin,
                class_wise_loo_margin,
            ) = compute_linear_svm_margin_scores(
                features=features,
                labels=labels,
                num_classes=NUM_CLASSES,
                C_svm=C_SVM,
            )

            results[model_name] = {
                "shots_per_class": shots,
                "total_samples": int(features.shape[0]),
                "overall_svm_margin": overall_margin,
                "overall_neg_sum_alpha": overall_neg_sum_alpha,
                "overall_loo_margin": overall_loo_margin,
                "class_wise_svm_margin": class_wise_margin,
                "class_wise_loo_margin": class_wise_loo_margin,
            }

        output_file = os.path.join(output_dir, "all_models_svm_margin.json")
        with open(output_file, "w") as f:
            json.dump(results, f, indent=4)

        print(f"Saved {shots}-shot scores to {output_file}")


Evaluating Linear Theta_0-SVM Margin (1-shot)...
Saved 1-shot scores to ./model_scores\model_scores_1_shot\SVM_margin_linear\all_models_svm_margin.json

Evaluating Linear Theta_0-SVM Margin (5-shot)...
Saved 5-shot scores to ./model_scores\model_scores_5_shot\SVM_margin_linear\all_models_svm_margin.json

Evaluating Linear Theta_0-SVM Margin (10-shot)...
Saved 10-shot scores to ./model_scores\model_scores_10_shot\SVM_margin_linear\all_models_svm_margin.json

Evaluating Linear Theta_0-SVM Margin (15-shot)...
Saved 15-shot scores to ./model_scores\model_scores_15_shot\SVM_margin_linear\all_models_svm_margin.json

Evaluating Linear Theta_0-SVM Margin (20-shot)...
Saved 20-shot scores to ./model_scores\model_scores_20_shot\SVM_margin_linear\all_models_svm_margin.json

Evaluating Linear Theta_0-SVM Margin (25-shot)...
Saved 25-shot scores to ./model_scores\model_scores_25_shot\SVM_margin_linear\all_models_svm_margin.json


## CLASSIFICATION NTK BASED SCORES

In [2]:
import os
import json
import itertools
import numpy as np
from sklearn.svm import SVC


# --- Configuration (Tunable) ---
SHOTS_LIST = [1, 5, 10, 15, 20, 25]
NUM_CLASSES = 6
NUM_SEEDS = 10       # S: number of random initializations averaged for LGC-CLS
C_SVM = 1.0          # Soft-margin penalty for Kernel SVM

root_prefix = ".." if os.path.exists("../model_embeddings_train") else "."
base_dir = os.path.join(root_prefix, "model_embeddings_train")
scores_root = os.path.join(root_prefix, "model_scores")


# --- Helper Functions ---
def get_first_k_subset(features: np.ndarray, labels: np.ndarray, shots_per_class: int):
    """
    Deterministically selects the first `shots_per_class` samples for each class
    in the order they appear in the archive.
    """
    selected_indices = []
    for c in np.unique(labels):
        c_indices = np.where(labels == c)[0]
        if len(c_indices) < shots_per_class:
            raise ValueError(
                f"Class {c} only has {len(c_indices)} samples; cannot pick {shots_per_class}."
            )
        selected_indices.extend(c_indices[:shots_per_class])

    selected_indices = np.array(selected_indices)
    return features[selected_indices], labels[selected_indices]


def _softmax(logits: np.ndarray) -> np.ndarray:
    shifted = logits - np.max(logits, axis=1, keepdims=True)
    exp_logits = np.exp(shifted)
    return exp_logits / np.sum(exp_logits, axis=1, keepdims=True)


def _build_scale_invariant_linear_ntk(features: np.ndarray) -> np.ndarray:
    """
    Builds the L2-normalized, double-centered, trace-normalized linear probe NTK:
        Theta_0 = Z_hat @ Z_hat.T + 1
        Theta_tilde = H_N @ Theta_0 @ H_N
        Theta_bar = (N / tr(Theta_tilde)) * Theta_tilde + 1e-9 * I_N
    """
    N = features.shape[0]
    X = features.astype(np.float64)
    norms = np.linalg.norm(X, axis=1, keepdims=True) + 1e-8
    X_norm = X / norms

    Theta_0 = (X_norm @ X_norm.T) + 1.0
    H_N = np.eye(N, dtype=np.float64) - (1.0 / N) * np.ones((N, N), dtype=np.float64)
    Theta_tilde = H_N @ Theta_0 @ H_N

    trace_val = np.trace(Theta_tilde)
    Theta_bar = (N / (trace_val + 1e-12)) * Theta_tilde

    # Enforce exact symmetry and add small jitter for QP stability
    Theta_bar = 0.5 * (Theta_bar + Theta_bar.T) + 1e-9 * np.eye(N, dtype=np.float64)
    return Theta_bar


def _fit_binary_precomputed_svm(K_sub: np.ndarray, y_sub: np.ndarray, C_svm: float = 1.0):
    """
    Fits a binary precomputed-kernel SVM on K_sub (M x M) with y_sub in {-1, +1}.
    Returns:
        clf: Fitted SVC object
        dual_obj: Wolfe dual objective W(alpha*) = sum(alpha) - 0.5 * ||f*||_H^2
        norm_f: RKHS norm ||f*||_H
    """
    clf = SVC(kernel="precomputed", C=C_svm)
    clf.fit(K_sub, y_sub)

    sv_indices = clf.support_
    y_alpha = clf.dual_coef_[0]  # Signed dual coefficients (y_i * alpha_i)
    sum_alpha = float(np.sum(np.abs(y_alpha)))

    K_sv = K_sub[np.ix_(sv_indices, sv_indices)]
    norm_sq = float(y_alpha @ K_sv @ y_alpha)
    norm_f = float(np.sqrt(max(norm_sq, 1e-12)))
    dual_obj = sum_alpha - 0.5 * norm_sq

    return clf, dual_obj, norm_f


# --- Metric 1: Classification-NTK LGC (LGC-CLS) ---
def compute_lgc_cls_linear(
    features: np.ndarray,
    labels: np.ndarray,
    num_classes: int = 6,
    num_seeds: int = 10,
):
    """
    Computes seed-averaged Classification-NTK LGC for a linear probe using
    unnormalized features Z:
        K_cls(i, j) = (z_i^T z_j + 1) * tr(H(p_i) H(p_j))
        LGC_cls = <K_cls_tilde, K_Y_tilde>_F / (||K_cls_tilde||_F * ||K_Y_tilde||_F)
    """
    N, D = features.shape
    Z = features.astype(np.float64)

    unique_classes, mapped_labels = np.unique(labels, return_inverse=True)
    if len(unique_classes) != num_classes:
        raise ValueError(f"Expected {num_classes} classes, got {len(unique_classes)}.")

    Y_onehot = np.eye(num_classes, dtype=np.float64)[mapped_labels]
    Y_tilde = Y_onehot - (1.0 / num_classes)
    K_Y = Y_tilde @ Y_tilde.T

    # Base linear logit kernel k_0(z_i, z_j) = z_i^T z_j + 1
    k0 = (Z @ Z.T) + 1.0
    I_C = np.eye(num_classes, dtype=np.float64)
    bound = 1.0 / np.sqrt(D)  # Matches PyTorch nn.Linear uniform init

    K_cls_sum = np.zeros((N, N), dtype=np.float64)

    for seed in range(num_seeds):
        rng = np.random.default_rng(seed)
        W = rng.uniform(-bound, bound, size=(D, num_classes))
        b = rng.uniform(-bound, bound, size=(1, num_classes))

        P = _softmax(Z @ W + b)  # (N, C)

        # Softmax Jacobians H_i = diag(p_i) - p_i p_i^T, shape (N, C, C)
        H = np.einsum("nc,cd->ncd", P, I_C) - np.einsum("nc,nd->ncd", P, P)

        # Since H_i is symmetric: tr(H_i H_j) = sum_{c,d} H_{i,cd} H_{j,cd}
        H_flat = H.reshape(N, num_classes * num_classes)
        trace_HH = H_flat @ H_flat.T

        K_cls_sum += k0 * trace_HH

    K_cls = K_cls_sum / num_seeds

    # Double-centering and Centered KTA
    H_N = np.eye(N, dtype=np.float64) - (1.0 / N) * np.ones((N, N), dtype=np.float64)
    K_cls_tilde = H_N @ K_cls @ H_N
    K_Y_tilde = H_N @ K_Y @ H_N

    norm_cls = np.linalg.norm(K_cls_tilde, ord="fro")
    norm_Y = np.linalg.norm(K_Y_tilde, ord="fro")
    denom = (norm_cls * norm_Y) + 1e-12

    Y_centered_cols = H_N @ Y_tilde  # (N, C)
    class_scores = {}
    overall_lgc_cls = 0.0

    for idx, c in enumerate(unique_classes):
        y_c = Y_centered_cols[:, idx]
        score_c = float(y_c.T @ K_cls_tilde @ y_c) / denom
        class_scores[int(c)] = score_c
        overall_lgc_cls += score_c

    return float(overall_lgc_cls), class_scores


# --- Metrics 2 & 3: In-Sample OvO Theta_0-SVM Margin & LOO Normalized Geometric Margin ---
def compute_svm_and_loo_margins_linear(
    features: np.ndarray,
    labels: np.ndarray,
    num_classes: int = 6,
    C_svm: float = 1.0,
):
    """
    Computes:
      - In-sample OvO Theta_0-SVM Margin: -mean_{c1 < c2} W^{(c1, c2)}(alpha*)
      - Leave-One-Out (LOO) Normalized Geometric Margin (K >= 2; None at K = 1)
    """
    N = features.shape[0]
    Theta_bar = _build_scale_invariant_linear_ntk(features)

    unique_classes = np.unique(labels)
    if len(unique_classes) != num_classes:
        raise ValueError(f"Expected {num_classes} classes, got {len(unique_classes)}.")

    class_indices = {c: np.where(labels == c)[0] for c in unique_classes}
    shots_per_class = len(class_indices[unique_classes[0]])

    # --- Metric 2: In-Sample OvO Theta_0-Margin ---
    pair_dual_objs = []
    class_margin_accum = {int(c): [] for c in unique_classes}

    for c1, c2 in itertools.combinations(unique_classes, 2):
        idx1 = class_indices[c1]
        idx2 = class_indices[c2]
        pair_idx = np.concatenate([idx1, idx2])
        y_pair = np.concatenate([
            np.ones(len(idx1), dtype=np.float64),
            -np.ones(len(idx2), dtype=np.float64),
        ])

        K_pair = Theta_bar[np.ix_(pair_idx, pair_idx)]
        _, dual_obj, _ = _fit_binary_precomputed_svm(K_pair, y_pair, C_svm=C_svm)

        pair_dual_objs.append(dual_obj)
        class_margin_accum[int(c1)].append(-dual_obj)
        class_margin_accum[int(c2)].append(-dual_obj)

    overall_svm_margin = -float(np.mean(pair_dual_objs))
    class_wise_svm_margin = {
        c: float(np.mean(vals)) for c, vals in class_margin_accum.items()
    }

    # --- Metric 3: Leave-One-Out (LOO) Normalized Geometric Margin ---
    if shots_per_class < 2:
        overall_loo_margin = None
        class_wise_loo_margin = None
    else:
        loo_margins = []
        class_loo_accum = {int(c): [] for c in unique_classes}

        for i in range(N):
            y_i = labels[i]
            idx_pos = np.array([idx for idx in class_indices[y_i] if idx != i])
            pairwise_margins_i = []

            for c_other in unique_classes:
                if c_other == y_i:
                    continue
                idx_neg = class_indices[c_other]
                train_idx = np.concatenate([idx_pos, idx_neg])
                y_train = np.concatenate([
                    np.ones(len(idx_pos), dtype=np.float64),
                    -np.ones(len(idx_neg), dtype=np.float64),
                ])

                K_train = Theta_bar[np.ix_(train_idx, train_idx)]
                clf, _, norm_f = _fit_binary_precomputed_svm(
                    K_train, y_train, C_svm=C_svm
                )

                # Signed functional margin of held-out sample i (+1 class)
                K_test_row = Theta_bar[np.ix_([i], train_idx)]
                f_i = float(clf.decision_function(K_test_row)[0])
                pairwise_margins_i.append(f_i / norm_f)

            worst_case_margin_i = float(np.min(pairwise_margins_i))
            loo_margins.append(worst_case_margin_i)
            class_loo_accum[int(y_i)].append(worst_case_margin_i)

        overall_loo_margin = float(np.mean(loo_margins))
        class_wise_loo_margin = {
            c: float(np.mean(vals)) for c, vals in class_loo_accum.items()
        }

    return (
        overall_svm_margin,
        class_wise_svm_margin,
        overall_loo_margin,
        class_wise_loo_margin,
    )


# --- Pipeline Execution (Notebook-Ready) ---
for shots in SHOTS_LIST:
    lgc_cls_dir = os.path.join(scores_root, f"model_scores_{shots}_shot", "LGC_cls_linear_new")
    svm_margin_dir = os.path.join(scores_root, f"model_scores_{shots}_shot", "SVM_margin_linear_new")
    loo_margin_dir = os.path.join(scores_root, f"model_scores_{shots}_shot", "SVM_loo_margin_linear_new")

    os.makedirs(lgc_cls_dir, exist_ok=True)
    os.makedirs(svm_margin_dir, exist_ok=True)
    os.makedirs(loo_margin_dir, exist_ok=True)

    lgc_cls_results = {}
    svm_margin_results = {}
    loo_margin_results = {}

    print(f"\n{'=' * 55}")
    print(f"Evaluating Linear New Scores ({shots}-shot | S={NUM_SEEDS} | C={C_SVM})")
    print(f"{'=' * 55}")

    for model_name in sorted(os.listdir(base_dir)):
        model_path = os.path.join(base_dir, model_name)
        if not os.path.isdir(model_path):
            continue

        npz_path = os.path.join(model_path, "features.npz")
        if not os.path.exists(npz_path):
            continue

        data = np.load(npz_path)
        try:
            features, labels = get_first_k_subset(
                data["embeddings"], data["labels"], shots_per_class=shots
            )
        except ValueError as e:
            print(f"  Skipping {model_name}: {e}")
            continue

        total_samples = int(features.shape[0])

        # 1. LGC-CLS
        overall_lgc_cls, class_wise_lgc_cls = compute_lgc_cls_linear(
            features=features,
            labels=labels,
            num_classes=NUM_CLASSES,
            num_seeds=NUM_SEEDS,
        )

        # 2 & 3. SVM Margin & LOO Margin
        (
            overall_svm_margin,
            class_wise_svm_margin,
            overall_loo_margin,
            class_wise_loo_margin,
        ) = compute_svm_and_loo_margins_linear(
            features=features,
            labels=labels,
            num_classes=NUM_CLASSES,
            C_svm=C_SVM,
        )

        lgc_cls_results[model_name] = {
            "shots_per_class": shots,
            "total_samples": total_samples,
            "num_seeds": NUM_SEEDS,
            "overall_lgc_cls_new": overall_lgc_cls,
            "class_wise_lgc_cls_new": class_wise_lgc_cls,
        }

        svm_margin_results[model_name] = {
            "shots_per_class": shots,
            "total_samples": total_samples,
            "C_svm": C_SVM,
            "overall_svm_margin_new": overall_svm_margin,
            "class_wise_svm_margin_new": class_wise_svm_margin,
        }

        loo_margin_results[model_name] = {
            "shots_per_class": shots,
            "total_samples": total_samples,
            "C_svm": C_SVM,
            "overall_loo_margin_new": overall_loo_margin,
            "class_wise_loo_margin_new": class_wise_loo_margin,
        }

        loo_str = f"{overall_loo_margin:.4f}" if overall_loo_margin is not None else "N/A (K=1)"
        print(
            f"  {model_name:<30} | LGC-CLS: {overall_lgc_cls:.4f} | "
            f"SVM-Margin: {overall_svm_margin:.4f} | LOO-Margin: {loo_str}"
        )

    lgc_file = os.path.join(lgc_cls_dir, "all_models_lgc_cls_new.json")
    svm_file = os.path.join(svm_margin_dir, "all_models_svm_margin_new.json")
    loo_file = os.path.join(loo_margin_dir, "all_models_svm_loo_margin_new.json")

    with open(lgc_file, "w") as f:
        json.dump(lgc_cls_results, f, indent=4)
    with open(svm_file, "w") as f:
        json.dump(svm_margin_results, f, indent=4)
    with open(loo_file, "w") as f:
        json.dump(loo_margin_results, f, indent=4)

    print(f"Saved {shots}-shot new scores to:\n  - {lgc_file}\n  - {svm_file}\n  - {loo_file}")


Evaluating Linear New Scores (1-shot | S=10 | C=1.0)
  AVES-bio                       | LGC-CLS: 0.9227 | SVM-Margin: -0.8972 | LOO-Margin: N/A (K=1)
  BirdAVES-biox-base             | LGC-CLS: 0.9526 | SVM-Margin: -0.8598 | LOO-Margin: N/A (K=1)
  DistilHuBERT                   | LGC-CLS: 0.7835 | SVM-Margin: -0.9499 | LOO-Margin: N/A (K=1)
  HuBERT                         | LGC-CLS: 0.8045 | SVM-Margin: -0.9266 | LOO-Margin: N/A (K=1)
  MusicHuBERT                    | LGC-CLS: 0.8128 | SVM-Margin: -0.8767 | LOO-Margin: N/A (K=1)
  WavLM                          | LGC-CLS: 0.7508 | SVM-Margin: -0.9784 | LOO-Margin: N/A (K=1)
Saved 1-shot new scores to:
  - .\model_scores\model_scores_1_shot\LGC_cls_linear_new\all_models_lgc_cls_new.json
  - .\model_scores\model_scores_1_shot\SVM_margin_linear_new\all_models_svm_margin_new.json
  - .\model_scores\model_scores_1_shot\SVM_loo_margin_linear_new\all_models_svm_loo_margin_new.json

Evaluating Linear New Scores (5-shot | S=10 | C=1.0)
  AV